<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 8: </b>RAG 평가</h2>
<br>

코스의 마지막 노트북에 오신 것을 환영합니다! 이전 노트북에서는 vector store 솔루션을 RAG 파이프라인에 통합했습니다! 이 노트북에서는 같은 파이프라인을 가져와 LLM-as-a-Judge 지표를 포함한 수치적 RAG 평가 기법으로 평가합니다!

<br>

### **학습 목표:**

- 이전 노트북들의 기법을 통합하여 RAG 파이프라인의 품질을 수치적으로 근사하는 방법을 배웁니다.

- **최종 실습**: ***코스 환경에서 이 노트북을 진행하면* 코스의 코딩 과제를 제출할 수 있습니다!**

<br>

### **생각해 볼 질문:**

- 진행하면서 우리의 지표가 실제로 무엇을 나타내는지 기억하세요. 우리 파이프라인이 이 목표를 통과해야 할까요? 우리의 judge LLM은 파이프라인을 평가하기에 충분할까요? 특정 지표가 우리 활용 사례에서 과연 중요할까요?
- 체인에 vectorstore-as-a-memory 구성 요소를 남겨 두었다면 여전히 평가를 통과할 것 같나요? 또한 이 평가가 vectorstore-as-a-memory 성능을 평가하는 데 유용한가요? 

<br>

### **환경 설정:**

In [ ]:
# %pip install -q langchain langchain-nvidia-ai-endpoints fastembed gradio rich
# %pip install -q arxiv pymupdf faiss-cpu ragas

## If you encounter a typing-extensions issue, restart your runtime and try again
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
norm_style = Style(bold=True)
pprint = partial(console.print, style=base_style)
pprint2 = partial(console.print, style=norm_style)

from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

embedder = NVIDIAEmbeddings(
    model="course/embedding",
    base_url="http://llm_client:9000/v1",
)

# In Colab, replace the service client above with:
# from langchain_community.embeddings import FastEmbedEmbeddings
# embedder = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ChatNVIDIA.get_available_models(base_url="http://llm_client:9000/v1")
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

----

<br>

## **Part 1:** 출시 전 평가

이전 노트북에서는 여러 개념을 성공적으로 결합하여, 반응성 있고 유익한 상호작용을 목표로 하는 문서 챗봇을 만들었습니다. 하지만 사용자 상호작용의 다양성 때문에 챗봇의 성능을 진정으로 이해하려면 포괄적인 테스트가 필요합니다. 시스템이 견고하고 다재다능할 뿐 아니라 사용자와 제공자의 기대에도 부합하도록 하려면 다양한 시나리오에서의 철저한 테스트가 필수적입니다.

챗봇의 역할을 정의하고 필요한 기능을 구현한 뒤에는, 평가가 다단계 과정이 됩니다:

- **일반 사용 점검:** 활용 사례에 가장 관련 있는 시나리오부터 테스트하세요. 챗봇이 사람의 개입을 최소화하면서 대화를 안정적으로 이끌어 갈 수 있는지 확인하세요.

    - 또한 사람에게 점검/감독을 위해 넘겨야 하는 한계나 영역(예: 거래 확인이나 민감한 탐색을 위한 사람 교체)을 파악하고 그 옵션을 구현하세요.

- **엣지 케이스 점검:** 일반 사용의 경계를 탐색하여, 덜 흔하지만 있을 법한 시나리오를 챗봇이 어떻게 처리하는지 파악하세요.

    - 공개 출시 전에는 부적절한 콘텐츠 생성 가능성처럼 법적 책임 위험을 초래할 수 있는 핵심 경계 조건을 평가하세요.

    - 원치 않는 상호작용을 제한하고 사용자를 예측 가능한 대화 흐름으로 유도하기 위해, 모든 출력(그리고 가능하면 입력)에 잘 테스트된 가드레일을 구현하세요.

- **점진적 출시:** 제한된 대상(먼저 내부, 그다음 [A/B](https://en.wikipedia.org/wiki/A/B_testing))에게 모델을 출시하고, 사용량 분석 대시보드와 피드백 채널(신고/좋아요/싫어요 등) 같은 분석 기능을 구현하세요.

이 세 단계 중 처음 두 단계는 소규모 팀이나 개인이 수행할 수 있으며 개발 과정의 일부로 반복되어야 합니다. 아쉽게도 이는 자주 수행해야 하고 사람의 실수에 취약할 수 있습니다. **다행히 LLM-as-a-Judge 방식으로 LLM의 도움을 받을 수 있습니다!**

*(네, 지금쯤이면 놀랍지 않을 겁니다. LLM이 강력하다는 것이 바로 이 코스가 존재하는 이유니까요...).*

----

<br>

## **Part 2:** LLM-as-a-Judge 방식

대화형 AI 영역에서 LLM을 평가자 또는 '심판(judge)'으로 사용하는 것은 자연어 작업 성능에 대한 설정 가능한 자동 테스트를 위한 유용한 접근법으로 떠올랐습니다:

- LLM은 다양한 상호작용 시나리오를 시뮬레이션하고 합성 데이터를 생성할 수 있으므로, 평가 개발자가 챗봇에서 다양한 동작을 이끌어 내기 위한 목표 지향적 입력을 생성할 수 있습니다.

- 합성 데이터에 대한 챗봇의 응답/검색은 LLM으로 평가하거나 파싱할 수 있으며, "Pass"/"Fail", 유사도, 추출 같은 일관된 출력 형식을 강제할 수 있습니다.

- 이런 결과를 다수 집계하여 "평가 통과율", "출처에서 가져온 관련 세부 정보의 평균 개수", "평균 코사인 유사도" 등을 설명하는 지표를 도출할 수 있습니다.

LLM을 사용해 챗봇 품질을 테스트하고 정량화한다는 이 아이디어는 [**"LLM-as-a-Judge"**](https://arxiv.org/abs/2306.05685)로 알려져 있으며, 사람의 판단과 밀접하게 일치하는 테스트 명세를 쉽게 만들 수 있고 대규모로 미세 조정하고 복제할 수 있습니다.

**기성 judge 방식을 위한 인기 있는 프레임워크에는 다음이 있습니다:**
- [**RAGAs(RAG Assessment의 약자)**](https://docs.ragas.io/en/stable/): 자체 평가 작업을 위한 훌륭한 출발점 모음을 제공합니다.
- [**LangChain Evaluators**](https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/): 암묵적으로 구성 가능한 많은 에이전트를 갖춘 유사한 자체 제공 옵션입니다.

체인을 그대로 사용하는 대신, 이 아이디어를 확장하여 더 커스텀한 솔루션으로 시스템을 평가하겠습니다.

----

<br>

## **Part 3: [평가 준비]** Pairwise Evaluator

다음 실습에서는 단순화된 [LangChain Pairwise String Evaluator](https://docs.langchain.com/langsmith/evaluate-pairwise)의 커스텀 구현을 구체화합니다. 

**RAG 체인 평가를 준비하기 위해 다음이 필요합니다:**

- 문서 인덱스(이전 노트북에서 저장한 것)를 가져옵니다.
- 선택한 RAG 파이프라인을 재구성합니다.

**구체적으로 다음 단계로 judge 방식을 구현합니다:**

- RAG 에이전트의 문서 풀에서 두 개의 문서 청크를 샘플링합니다.
- 그 두 문서 청크를 사용해 합성 "기준(baseline)" 질문-답변 쌍을 생성합니다.
- RAG 에이전트를 사용해 자체 답변을 생성합니다.
- 합성 생성물을 "정답(ground-truth)"으로 놓고 judge LLM으로 두 응답을 비교합니다.

**이 체인은 다음 목표를 테스트하는 단순하지만 강력한 과정이어야 합니다:**

> ***내 RAG 체인이 문서 접근이 제한된 좁은 범위의 챗봇보다 더 나은 성능을 내는가.***





**이것이 최종 평가에 사용되는 시스템입니다!** 이 시스템이 자동 채점기에 어떻게 통합되는지 보려면 [`frontend/frontend_block.py`](frontend/frontend_block.py)를 확인하세요.

<br>

### **Task 1:** 문서 검색 인덱스 가져오기

이 실습에서는 이전 노트북에서 만든 `docstore_index` 파일을 가져옵니다. 아래 셀로 스토어를 그대로 로드할 수 있어야 합니다.

In [ ]:
## Make sure you have docstore_index.tgz in your working directory
from langchain_community.vectorstores import FAISS

!tar xzvf docstore_index.tgz
docstore = FAISS.load_local("docstore_index", embedder, allow_dangerous_deserialization=True)
docs = list(docstore.docstore._dict.values())

def format_chunk(doc):
    return (
        f"Paper: {doc.metadata.get('Title', 'unknown')}"
        f"\n\nSummary: {doc.metadata.get('Summary', 'unknown')}"
        f"\n\nPage Body: {doc.page_content}"
    )

## This printout just confirms that your store has been retrieved
pprint(f"Constructed aggregate docstore with {len(docstore.docstore._dict)} chunks")
pprint(f"Sample Chunk:")
print(format_chunk(docs[len(docs)//2]))

<br>

### **Task 2: [실습]** RAG Chain 가져오기

인덱스를 얻었으니, 이전 노트북의 RAG 에이전트를 다시 만들 수 있습니다! 

**주요 수정 사항:**
- 단순함을 위해 vectorstore-as-a-memory 구성 요소는 무시해도 좋습니다. 이를 포함하면 오버헤드가 더 필요하고 실습이 조금 더 복잡해집니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_community.document_transformers import LongContextReorder

from langchain_nvidia_ai_endpoints import ChatNVIDIA

from functools import partial
from operator import itemgetter

import gradio as gr

#####################################################################


# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})
llm = instruct_llm | StrOutputParser()

#####################################################################

def docs2str(docs, title="Document"):
    """Useful utility for making chunks into context string. Optional, but useful"""
    out_str = ""
    for doc in docs:
        doc_name = getattr(doc, 'metadata', {}).get('Title', title)
        if doc_name: out_str += f"[Quote from {doc_name}] "
        out_str += getattr(doc, 'page_content', str(doc)) + "\n"
    return out_str

chat_prompt = ChatPromptTemplate.from_template(
    "You are a document chatbot. Help the user as they ask questions about documents."
    " User messaged just asked you a question: {input}\n\n"
    " The following information may be useful for your response: "
    " Document Retrieval:\n{context}\n\n"
    " (Answer only from retrieval. Only cite sources that are used. Make your response conversational)"
    "\n\nUser Question: {input}"
)

def output_puller(inputs):
    """"Output generator. Useful if your chain returns a dictionary with key 'output'"""
    if isinstance(inputs, dict):
        inputs = [inputs]
    for token in inputs:
        if token.get('output'):
            yield token.get('output')

#####################################################################
## TODO: Pull in your desired RAG Chain. Memory not necessary

## Chain 1 Specs: "Hello World" -> retrieval_chain 
##   -> {'input': <str>, 'context' : <str>}
long_reorder = RunnableLambda(LongContextReorder().transform_documents)  ## GIVEN
context_getter = RunnableLambda(lambda x: x)  ## TODO
retrieval_chain = {'input' : (lambda x: x)} | RunnableAssign({'context' : context_getter})

## Chain 2 Specs: retrieval_chain -> generator_chain 
##   -> {"output" : <str>, ...} -> output_puller
generator_chain = RunnableLambda(lambda x: x)  ## TODO
generator_chain = {'output' : generator_chain} | RunnableLambda(output_puller)  ## GIVEN

## END TODO
#####################################################################

rag_chain = retrieval_chain | generator_chain

# pprint(rag_chain.invoke("Tell me something interesting!"))
for token in rag_chain.stream("Tell me something interesting!"):
    print(token, end="")

<br>

### **Step 3:** 합성 질문-답변 쌍 생성하기

이 섹션에서는 평가 루틴의 처음 몇 부분을 구현합니다:

- **RAG 에이전트의 문서 풀에서 두 개의 문서 청크를 샘플링합니다.**
- **그 두 문서 청크를 사용해 합성 "기준" 질문-답변 쌍을 생성합니다.**
- RAG 에이전트를 사용해 자체 답변을 생성합니다.
- 합성 생성물을 "정답"으로 놓고 judge LLM으로 두 응답을 비교합니다.

이 체인은 다음 목표를 테스트하는 단순하지만 강력한 과정이어야 합니다:

> 내 RAG 체인이 문서 접근이 제한된 좁은 범위의 챗봇보다 더 나은 성능을 내는가?

In [ ]:
import random
import re

num_questions = 3
synth_questions = []
synth_answers = []

simple_prompt = ChatPromptTemplate.from_messages([('system', '{system}'), ('user', 'INPUT: {input}')])

for i in range(num_questions):
    doc1, doc2 = random.sample(docs, 2)
    sys_msg = (
        "Use the documents provided by the user to generate an interesting question-answer pair."
        " Try to use both documents if possible, and rely more on the document bodies than the summary."
        " Use the format:\nQuestion: (good question, 1-3 sentences, detailed)\n\nAnswer: (answer derived from the documents)"
        " DO NOT SAY: \"Here is an interesting question pair\" or similar. FOLLOW FORMAT!"
    )
    usr_msg = (
        f"Document1: {format_chunk(doc1)}\n\n"
        f"Document2: {format_chunk(doc2)}"
    )

    qa_pair = (simple_prompt | llm).invoke({'system': sys_msg, 'input': usr_msg})
    match = re.search(r"(?:(?:\*\*)?(?:Question|質問|问题|問題)(?:\*\*)?\s*[:：](?:\*\*)?\s*)?(.*?)\s*(?:\*\*)?(?:Answer|回答|答案)(?:\*\*)?\s*[:：](?:\*\*)?\s*(.*)", qa_pair, flags=re.DOTALL)
    if not match:
        raise ValueError("The generated response did not contain an Answer section. Please rerun this cell.")
    synth_questions += ["Question: " + match.group(1).strip()]
    synth_answers += ["Answer: " + match.group(2).strip()]
    pprint2(f"QA Pair {i+1}")
    pprint2(synth_questions[-1])
    pprint(synth_answers[-1])
    print()

<br>

### **Step 4:** 합성 질문에 답하기

이 섹션에서는 평가 루틴의 세 번째 부분을 구현합니다:

- RAG 에이전트의 문서 풀에서 두 개의 문서 청크를 샘플링합니다.
- 그 두 문서 청크를 사용해 합성 "기준" 질문-답변 쌍을 생성합니다.
- **RAG 에이전트를 사용해 자체 답변을 생성합니다.**
- 합성 생성물을 "정답"으로 놓고 judge LLM으로 두 응답을 비교합니다.

이 체인은 다음 목표를 테스트하는 단순하지만 강력한 과정이어야 합니다:

> 내 RAG 체인이 문서 접근이 제한된 좁은 범위의 챗봇보다 더 나은 성능을 내는가?

In [ ]:
## TODO: Generate some synthetic answers to the questions above.
##   Try to use the same syntax as the cell above
rag_answers = []
for i, q in enumerate(synth_questions):
    ## TODO: Compute the RAG Answer
    rag_answer = ""
    rag_answers += [rag_answer]
    pprint2(f"QA Pair {i+1}", q, "", sep="\n")
    pprint(f"RAG Answer: {rag_answer}", "", sep='\n')

<br>

### **Step 5:** 사람 선호 지표 구현하기

이 섹션에서는 평가 루틴의 네 번째 부분을 구현합니다:

- RAG 에이전트의 문서 풀에서 두 개의 문서 청크를 샘플링합니다.
- 그 두 문서 청크를 사용해 합성 "기준" 질문-답변 쌍을 생성합니다.
- RAG 에이전트를 사용해 자체 답변을 생성합니다.
- **합성 생성물을 "정답"으로 놓고 judge LLM으로 두 응답을 비교합니다.**

이 체인은 다음 목표를 테스트하는 단순하지만 강력한 과정이어야 합니다:

> 내 RAG 체인이 문서 접근이 제한된 좁은 범위의 챗봇보다 더 나은 성능을 내는가?

In [ ]:
## TODO: Adapt this prompt for whichever LLM you're actually interested in using. 
## If it's llama, maybe system message would be good?
eval_prompt = ChatPromptTemplate.from_template("""INSTRUCTION: 
Evaluate the following Question-Answer pair for human preference and consistency.
Assume the first answer is a ground truth answer and has to be correct.
Assume the second answer may or may not be true.
[1] The second answer lies, does not answer the question, or is inferior to the first answer.
[2] The second answer is better than the first and does not introduce any inconsistencies.

Output Format:
[Score] Justification

{qa_trio}

EVALUATION: 
""")

pref_score = []

trio_gen = zip(synth_questions, synth_answers, rag_answers)
for i, (q, a_synth, a_rag) in enumerate(trio_gen):
    pprint2(f"Set {i+1}\n\nQuestion: {q}\n\n")

    qa_trio = f"Question: {q}\n\nAnswer 1 (Ground Truth): {a_synth}\n\n Answer 2 (New Answer): {a_rag}"
    pref_score += [(eval_prompt | llm).invoke({'qa_trio': qa_trio})]
    pprint(f"Synth Answer: {a_synth}\n\n")
    pprint(f"RAG Answer: {a_rag}\n\n")
    pprint2(f"Synth Evaluation: {pref_score[-1]}\n\n")

<br>

**축하합니다! 이제 우리 파이프라인에 대해 추론하고 평가하려는 LLM 시스템을 갖추었습니다!** judge 결과를 얻었으니, 결과를 단순히 집계하여 LLM 기준으로 우리 방식이 얼마나 자주 통과했는지 확인할 수 있습니다:

In [ ]:
pref_score = sum(("[2]" in score) for score in pref_score) / len(pref_score)
print(f"Preference Score: {pref_score}")

----

<br>

## **Part 4:** 고급 방식

위 실습은 코스의 최종 평가를 준비하기 위한 것으로, 단순하지만 효과적인 평가자 체인을 보여 주었습니다. 목표와 구현 세부 사항이 제공되었으며, 실제 동작을 보았으니 이를 사용하는 논리도 이제 이해가 될 것입니다. 

그렇지만 이 지표는 단지 우리가 다음을 명시한 결과물일 뿐입니다:
- **우리 파이프라인이 갖추어야 할 중요한 동작은 무엇인가?**
- **이 동작을 드러내고 평가하려면 무엇을 해야 하는가?**

이 두 질문으로부터, 다른 속성을 평가하고, 다른 평가자 체인 기법을 통합하고, 심지어 다른 파이프라인 구성 전략을 요구하는 다양한 다른 평가 지표를 만들어 낼 수 있었을 것입니다. 전부는 아니지만, 자주 마주칠 만한 일반적인 방식은 다음과 같습니다:

- **스타일 평가:** 어떤 평가 방식은 "질문을 몇 개 던지고 출력이 바람직하게 느껴지는지 본다"처럼 단순할 수 있습니다. 이는 judge LLM에 제공된 설명을 기준으로 챗봇이 "의도된 대로 행동하는지" 확인하는 데 사용될 수 있습니다. 이런 평가는 프롬프트 엔지니어링과 while 루프만으로도 충분히 달성할 수 있기 때문에 따옴표를 썼습니다.

- **정답 기반 평가:** 우리 체인에서는 샘플링 전략을 사용한 합성 생성으로 무작위 질문과 답변을 만들었지만, 실제로는 챗봇이 일관되게 맞혀야 하는 대표적인 질문과 답변을 갖고 있을 수 있습니다! 이 경우 위 실습 체인을 수정하여 구현하고, 파이프라인을 개발하면서 면밀히 모니터링해야 합니다.

- **검색/증강 평가:** 이 코스는 파이프라인에 어떤 전처리와 프롬프팅 단계가 좋을지에 대해 많은 가정을 했고, 그 대부분은 실험으로 결정되었습니다. 문서 전처리, 청킹 전략, 모델 선택, 프롬프트 명세 같은 요소들이 모두 중요한 역할을 했으므로, 이런 결정을 검증하는 지표를 만드는 데 관심이 있을 수 있습니다. 이런 종류의 지표는 파이프라인이 컨텍스트 청크를 출력해야 할 수도 있고, 임베딩 유사도 비교에만 의존할 수도 있으므로, 여러 평가 전략과 함께 동작하는 체인을 구현할 때 이를 염두에 두세요. 커스텀 일반화 가능 평가 루틴을 만들기 위한 괜찮은 출발점으로 [**RagasEvaluatorChain**](https://docs.ragas.io/en/v0.1.21/howtos/integrations/langchain.html) 추상화를 고려해 보세요. 

- **궤적(Trajectory) 평가:** 더 고급 에이전트 방식을 사용하면 대화 메모리의 존재를 가정하는 다중 쿼리 전략을 구현할 수 있습니다. 이를 통해 다음을 수행하는 평가 에이전트를 구현할 수 있습니다:
    - 에이전트가 시나리오에 얼마나 잘 적응하고 대응하는지 평가하기 위해 일련의 질문을 던집니다. 이런 시스템은 일반적으로 일련의 응답을 고려하며, 에이전트가 대화를 어떻게 이끌어 갔는지의 "궤적"을 끌어내어 평가하는 것을 목표로 합니다. [**LangChain Trajectory Evaluators 문서**](https://docs.langchain.com/langsmith/trajectory-evals)가 좋은 출발점입니다.
    - 또는 챗봇과 상호작용하며 목표를 달성하려 하는 평가 에이전트를 구현할 수도 있습니다. 이런 에이전트는 자연스러운 방식으로 해결책에 도달할 수 있었는지 출력할 수 있으며, 체감 성능에 대한 보고서를 생성하는 데도 사용할 수 있습니다. [**LangChain Agents 문서**](https://docs.langchain.com/oss/python/langchain/agents)가 좋은 출발점입니다!

<br>

결국 중요한 것은 가진 도구를 적절히 사용하는 것입니다. 코스의 이 시점에서는 이미 LLM의 핵심 가치 제안에 익숙해져 있어야 합니다: **강력하고, 확장 가능하고, 예측 가능하고, 제어 가능하고, 오케스트레이션 가능하지만... 기본적으로 그냥 잘 동작하리라 기대하면 예측 불가능하게 행동합니다.** 필요를 평가하고, 파이프라인을 구성하고 검증하고, 충분한 정보를 제공하고, 시스템이 일관되고 효율적이며 효과적으로 동작하도록 가능한 한 많은 제어를 추가하세요.

----

<br>

## **Part 5: [평가]** 학점 인정을 위한 평가

코스의 마지막 실습에 오신 것을 환영합니다! 내용을 즐기셨고, 이제 이 노트북들에 대해 실제로 학점을 받을 준비가 되셨기를 바랍니다! 이 부분에서는:

- **코스 환경에 있는지 확인하세요**
- **`docstore_index/`가 코스 환경에 업로드되어 있는지 확인하세요...**
- **[`09_langserve.ipynb`](09_langserve.ipynb)의 오래된 세션이 이미 포트를 점유하고 있지 않은지 확인하세요. 평가에서는 새로운 `/retriever`와 `/generator` 엔드포인트를 구현해야 합니다!!**

**목표:** 실행 시 프론트엔드는 [**`frontend/frontend_block.py`**](frontend/frontend_block.py)에서 평가 로직을 로드합니다. 시작 후 코스 환경은 실행 중인 평가기를 바꾸지 않고 소스의 통과 마커 표현식을 정리합니다. 여러분의 목표는 파이프라인을 사용해 **Evaluation** 검사를 통과하는 것입니다! [`09_langserve.ipynb`](09_langserve.ipynb)를 떠올리고 이를 시작 예제로 사용하세요. 원본을 권위 있는 참고 자료로 유지할 수 있도록 복제해서 작업하는 것을 권장합니다. 

**완료 후:** 코스 환경이 아직 열려 있는 동안, 코스 환경 런처 영역으로 돌아가 **"Assess Task"** 버튼을 클릭하세요! 그러면 모두 끝입니다!

<a href="/8090" target="_blank" style="display: inline-block; padding: 12px 24px; background-color: #76b900; color: white; text-decoration: none; border-radius: 4px; font-weight: bold; margin: 3px;">Gradio Frontend UI (<code>/8090</code> -> <code>:8090</code> 안정성을 위해)</a>

In [ ]:
# %%js
# // Manual Access Without NGINX
# var url = 'http://'+window.location.host+':8090';
# element.innerHTML = '<a style="color:green;" target="_blank" href='+url+'><h1>< Link To Gradio Frontend ></h1></a>';

----

<br>

## <font color="#76b900">**코스 완료를 축하합니다**</font>

이 코스가 흥미롭고 도전적이었을 뿐 아니라, LLM 및 RAG 시스템 개발의 최전선에서 일할 수 있도록 충분히 준비시켜 주었기를 바랍니다! 앞으로 여러분은 업계 수준의 과제를 다루고, 오픈소스 모델과 프레임워크로 RAG 배포를 탐구하는 데 필요한 역량을 갖추게 되었을 것입니다.

**이와 관련하여 흥미로울 만한 NVIDIA 관련 릴리스는 다음과 같습니다:**
- [**NVIDIA NIM**](https://www.nvidia.com/en-us/ai-data-science/products/nim-microservices/): 로컬 컴퓨팅에 배포할 수 있는 마이크로서비스 실행 루틴을 제공합니다.
- [**TensorRT-LLM**](https://github.com/NVIDIA/TensorRT-LLM): 프로덕션 환경에서 GPU 가속 LLM 모델 엔진을 배포하기 위해 현재 권장되는 프레임워크입니다.
- [**NVIDIA Generative AI Examples 저장소**](https://github.com/NVIDIA/GenerativeAIExamples): 현재 표준 마이크로서비스 예제 애플리케이션을 포함하며, 새로운 프로덕션 워크플로가 출시됨에 따라 새 리소스로 업데이트됩니다.

**또한 더 깊이 파고들어 볼 만한 주요 주제는 다음과 같습니다:**
- [**LlamaIndex**](https://www.llamaindex.ai/): LangChain의 RAG 기능을 보강하고 때로는 개선할 수 있는 강력한 구성 요소를 갖추고 있습니다.
- [**LangSmith**](https://docs.smith.langchain.com/): LangChain이 제공하는 에이전트 프로덕션화 서비스입니다.
- [**Gradio**](https://www.gradio.app/): 코스에서 다루긴 했지만 살펴볼 만한 인터페이스 옵션이 훨씬 많습니다. 영감을 얻으려면 [**HuggingFace Spaces**](https://huggingface.co/spaces)의 예시를 참고하세요.
- [**LangGraph**](https://python.langchain.com/docs/langgraph/): 그래프 기반 LLM 오케스트레이션 프레임워크로, [멀티 에이전트 워크플로](https://blog.langchain.dev/langgraph-multi-agent-workflows/)에 관심 있는 분들에게 자연스러운 다음 단계입니다.
- [**DSPy**](https://github.com/stanfordnlp/dspy): 경험적 성능 결과를 바탕으로 LLM 오케스트레이션 파이프라인을 최적화할 수 있는 플로 엔지니어링 프레임워크입니다.

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>